In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Step 1: Create an imbalanced binary classification dataset
X, y = make_classification(n_samples=1000, n_features=10, n_informative=2, n_redundant=8, 
                           weights=[0.9, 0.1], flip_y=0, random_state=42)
np.unique(y, return_counts=True)

(array([0, 1]), array([900, 100], dtype=int64))

In [5]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

### Handle class imbalance

In [12]:
from imblearn.combine import SMOTETomek

smt = SMOTETomek(random_state=42)
X_train_res, y_train_res = smt.fit_resample(X_train, y_train)
np.unique(y_train_res, return_counts=True)

(array([0, 1]), array([619, 619], dtype=int64))

### Track Experiments

In [16]:
models = [
    (
        "Logistic Regression", 
        {"C": 1, "solver": 'liblinear'},
        LogisticRegression(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Random Forest", 
        {"n_estimators": 30, "max_depth": 3},
        RandomForestClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier With SMOTE",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train_res, y_train_res),
        (X_test, y_test)
    )
]


In [18]:
reports = []

for model_name, params, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]
    
    model.set_params(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    reports.append(report)

In [20]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

In [23]:
# Initialize MLflow
mlflow.set_experiment("Anomaly_Detection_1")
mlflow.set_tracking_uri("http://localhost:5000")

for i, element in enumerate(models):
    model_name = element[0]
    params = element[1]
    model = element[2]
    report = reports[i]
    
    with mlflow.start_run(run_name=model_name):        
        mlflow.log_params(params)
        mlflow.log_metrics({
            'accuracy': report['accuracy'],
            'recall_class_1': report['1']['recall'],
            'recall_class_0': report['0']['recall'],
            'f1_score_macro': report['macro avg']['f1-score']
        })  
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")  


2025/05/01 17:33:53 INFO mlflow.tracking.fluent: Experiment with name 'Anomaly_Detection_1' does not exist. Creating a new experiment.
2025/05/01 17:34:02 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run Logistic Regression at: http://localhost:5000/#/experiments/639198555058823232/runs/44722b6c1a2848e4a76cfd92d6f7708b
🧪 View experiment at: http://localhost:5000/#/experiments/639198555058823232


2025/05/01 17:34:12 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run Random Forest at: http://localhost:5000/#/experiments/639198555058823232/runs/636e5987c7644e39928cc28c83bf3873
🧪 View experiment at: http://localhost:5000/#/experiments/639198555058823232


2025/05/01 17:34:27 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run XGBClassifier at: http://localhost:5000/#/experiments/639198555058823232/runs/cbf3d3c4426a4f6ea78e67a4b6d9c044
🧪 View experiment at: http://localhost:5000/#/experiments/639198555058823232


2025/05/01 17:34:42 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run XGBClassifier With SMOTE at: http://localhost:5000/#/experiments/639198555058823232/runs/e38aca137e0344c292b27aef5d3c8384
🧪 View experiment at: http://localhost:5000/#/experiments/639198555058823232


### Register the Model

In [28]:
model_name='XGB-Smote'
run_id=input('enter runid')
model_uri=f'runs:/{run_id}/model_name'
with mlflow.start_run():
    mlflow.register_model(model_uri=model_uri,name=model_name)

enter runid e38aca137e0344c292b27aef5d3c8384


Successfully registered model 'XGB-Smote'.
2025/05/01 17:41:25 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB-Smote, version 1


🏃 View run receptive-shrimp-257 at: http://localhost:5000/#/experiments/639198555058823232/runs/507011dd91404980a8fd3896bbddf90b
🧪 View experiment at: http://localhost:5000/#/experiments/639198555058823232


Created version '1' of model 'XGB-Smote'.


### Load the Model

In [ ]:
model_version=1
model_uri=f"models:/{model_name}@challenger"

loded=mlflow.xgboost.load_model(model_uri)
y_pred=loded.predict(X_test)
y_pred[:4]

### Transition the Model to Production

In [43]:
dev_model=f"models:/{model_name}@challenger"
prod_model='anomaly_detection_1-product'

client=mlflow.MlflowClient()
client.copy_model_version(src_model_uri=dev_model,dst_name=prod_model)

Successfully registered model 'anomaly_detection_1-product'.
Copied version '1' of model 'XGB-Smote' to version '1' of model 'anomaly_detection_1-product'.


<ModelVersion: aliases=[], creation_timestamp=1746104210632, current_stage='None', description='', last_updated_timestamp=1746104210632, name='anomaly_detection_1-product', run_id='e38aca137e0344c292b27aef5d3c8384', run_link='', source='models:/XGB-Smote/1', status='READY', status_message=None, tags={}, user_id='', version='1'>

In [ ]:
model_version=1
model_uri=f"models:/{prod_model}@champion"

loded=mlflow.xgboost.load_model(model_uri)
y_pred=loded.predict(X_test)
y_pred[:2]